# Shadow-1-Whisper-0.8B — Fine-Tune & Export (free Colab T4)

Fine-tunes **`unsloth/Qwen3.5-0.8B`** on ShadowTalk's SFT dataset (`shadowtalk_sft.jsonl`, 620
examples: 45% shadowvisual JSON / 25% agentic missions / 20% grounded RAG / 10% identity+multilingual)
and exports **three artifacts**:

| Artifact | Repo | Consumer |
|---|---|---|
| Merged 16-bit weights | `shadowtalkai/Shadow-1-Whisper-0.8B` | base for GGUF + ONNX transplant |
| GGUF `q4_k_m` | `shadowtalkai/Shadow-1-Whisper-0.8B-GGUF` | Tauri Desktop (llama.cpp) |
| ONNX q4 (WebGPU) | `shadowtalkai/Shadow-1-Whisper-0.8B-ONNX` | Browser worker (`@huggingface/transformers` v4+) |

**Before running:** get a Hugging Face **write** token (fine-grained, repos: *write*), create the three
repos under `shadowtalkai/` (or let Cell 4 create them), and upload `shadowtalk_sft.jsonl` next to this
notebook. Everything below runs on a free T4 (~3 GB VRAM for LoRA, fp16).

## Cell 1 — Setup
Fails fast if the runtime has no GPU, then installs the stack **exactly as the official
unsloth Qwen3.5 Vision Colab notebook does** — pinned `torch 2.8.0`, `transformers==5.2.0`,
`trl==0.22.2`, `tokenizers≤0.23.0`, xformers + `unsloth install-kernels` — plus the
export-side ONNX tools. Takes ~3-5 min on a fresh Colab runtime.

In [ ]:
# 1 ─ Setup — fail-fast GPU check, then the PINNED install stack
#
# Design notes (why this cell is shaped this way):
#
# torch / torchvision / torchaudio / xformers are COMPILED against each other.
# The torchvision::nms operator lives in torchvision/_C.so, which only loads
# when its ABI matches the installed torch build exactly — Colab's preinstalled
# pair is matched, but ANY mixed pair (PyPI torchvision over a custom torch
# build, or downgrading one side without the other) makes _C.so fail to dlopen.
# torch's extension loader swallows that failure, and the first visible
# symptom is Cell 2 dying inside unsloth with:
#     RuntimeError: operator torchvision::nms does not exist
# So this cell installs the stack as ONE matched set (single joint uv
# resolution — never torch and torchvision in separate commands) and then
# VERIFIES the compiled ops actually load, in a fresh subprocess, at setup
# time — instead of failing half an hour later inside unsloth's import.
#
# Never reinstall into a session that already imported the stack: binaries
# replaced under a live interpreter leave the process mixed with the on-disk
# state and reproduce the same nms crash. Only a runtime restart clears that;
# reinstalling cannot. (That is also why the GPU check below uses nvidia-smi
# in a subprocess, NOT `import torch` — importing torch here would lock the
# preinstalled build into this process before the pinned install lands.)
import sys, subprocess, importlib.util, json

_already = [m for m in ("torch", "unsloth", "unsloth_zoo", "torchvision", "transformers", "trl", "peft") if m in sys.modules]
assert not _already, (
    f"{', '.join(_already)} already imported in this session — re-running the installer now would "
    "corrupt the loaded binary state. Runtime → Restart session, then re-run this cell."
)
_gpu = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,compute_cap", "--format=csv,noheader"],
    capture_output=True, text=True,
)
assert _gpu.returncode == 0 and _gpu.stdout.strip(), (
    "nvidia-smi found no GPU — Runtime → Change runtime type → T4 GPU required"
)
print(f"GPU: {_gpu.stdout.strip().splitlines()[0]}")

# The pinned stack — the EXACT combo the official unsloth Qwen3.5 Vision Colab
# notebook (unslothai/notebooks · nb/Qwen3_5_(0_8B)_Vision.ipynb) installs for
# our base model. The pins are load-bearing: PyPI's latest transformers is
# 5.24.x, but unsloth's Qwen3.5 kernels are built and tested against
# transformers==5.2.0 + trl==0.22.2 + tokenizers<=0.23.0 — a loose
# 'transformers>=5.2.0' resolves to an untested pairing and breaks the patched
# forward pass / SFTConfig API.
_TORCH_PIN, _TV_PIN = "2.8.0", "0.23.0"

# Probe: a FRESH interpreter imports torch + torchvision and loads the
# compiled nms op. torch.__version__ carries a local tag ('2.8.0+cu128'), so
# pin checks use startswith, not equality.
_PROBE_SRC = r'''
import json, torch, torchvision
from torchvision.ops import nms
ok = nms(torch.tensor([[0., 0., 1., 1.], [0., 0., 2., 2.]]),
         torch.tensor([0.9, 0.8]), 0.5)
print(json.dumps({"torch": torch.__version__, "torchvision": torchvision.__version__, "nms": ok.numel() == 2}))
'''
def _probe():
    r = subprocess.run([sys.executable, "-c", _PROBE_SRC], capture_output=True, text=True)
    for line in reversed((r.stdout or "").strip().splitlines()):
        if line.startswith("{"):
            try:
                return json.loads(line)
            except Exception:
                return None
    return None

state = _probe()
_matched = (
    state is not None
    and state["torch"].startswith(_TORCH_PIN)
    and state["torchvision"].startswith(_TV_PIN)
)

# ── fail-fast installer ──────────────────────────────────────────────
# IPython's `!` shell lines NEVER raise on nonzero exit: a failed install
# cascades invisibly and surfaces much later as a misleading assert. Every
# setup command goes through _run() so a failure stops the cell at the step
# that actually failed, with its command in the message.
#
# uv must install into the interpreter that RUNS THIS KERNEL — not whatever
# environment uv's ambient discovery picks (a stray VIRTUAL_ENV, UV_PYTHON
# or uv.toml silently redirects it; the redirected install 'succeeds' while
# nothing lands in the kernel's environment). Pin the target explicitly.
_IN_VENV = sys.prefix != getattr(sys, "base_prefix", sys.prefix)
_UV_ENV = f'--python "{sys.executable}"' if _IN_VENV else "--system"
_PY_ENV = f'"{sys.executable}" -m pip' if _IN_VENV else "python3 -m pip"

def _run(cmd, fatal=True):
    r = subprocess.run(cmd, shell=True)
    if fatal:
        assert r.returncode == 0, f"setup step failed (exit {r.returncode}): {cmd}"
    return r

_run(f"{_PY_ENV} install --upgrade -qqq uv")  # official pattern; always before any uv call

if not _matched:
    if state is None:
        print("torch/torchvision missing or compiled ops broken — installing the pinned ABI-matched set (~4-6 min on a fresh runtime)…")
    else:
        print(f"torch {state['torch']} / torchvision {state['torchvision']} ≠ pinned {_TORCH_PIN}/{_TV_PIN} — replacing as ONE matched set (~4-6 min)…")
    try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
    except Exception: _numpy = "numpy"; _pil = "pillow"
    # ONE joint resolution: uv picks the torchvision that matches torch==2.8.0 —
    # never a mixed pair. (The official notebook resolves it exactly this way.)
    _run(
        f'uv pip install -qqq {_UV_ENV} '
        f'"torch=={_TORCH_PIN}" "triton>=3.3.0" {_numpy} {_pil} torchvision '
        f'"torchaudio=={_TORCH_PIN}" bitsandbytes xformers==0.0.32.post2 '
        '"unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" '
        '"unsloth[base] @ git+https://github.com/unslothai/unsloth"'
    )
    _run(f'uv pip install -qqq {_UV_ENV} --no-deps "torchcodec==0.7.0"')
else:
    print(f"pinned stack already on disk (torch {state['torch']}, torchvision {state['torchvision']}) — skipping the heavy install")
    _missing = [p for p in ("unsloth", "unsloth_zoo", "xformers", "bitsandbytes") if importlib.util.find_spec(p) is None]
    if _missing:
        _run(f'uv pip install -qqq {_UV_ENV} unsloth unsloth_zoo xformers==0.0.32.post2 bitsandbytes')

# Force the exact library versions unsloth's Qwen3.5 kernels were tested
# against (lines identical to the official notebook; the git-HEAD installs
# above are never silently downgraded — uv keeps whichever version is newer).
_run(f'uv pip install {_UV_ENV} --upgrade --no-deps "tokenizers>=0.22.0,<=0.23.0" trl==0.22.2 unsloth unsloth_zoo')
_run(f'uv pip install {_UV_ENV} transformers==5.2.0')
_run("unsloth install-kernels")
_run(f'uv pip install {_UV_ENV} --no-deps --upgrade "torchao>=0.16.0"')

# Export-side tools for Cell 4/5 (ONNX graph verify + tokenizer export). Deliberately
# WITHOUT -U so nothing drags huggingface_hub/safetensors away from what
# transformers==5.2.0 pins.
if any(importlib.util.find_spec(p) is None for p in ("onnx", "onnxruntime", "onnx_ir", "sentencepiece")):
    _run(f"{_PY_ENV} install -q onnx onnxruntime onnx_ir sentencepiece protobuf")  # onnx_ir: MatMulNBits quantizer dep (ORT >=1.20)

# Final gate: the compiled ops MUST load in a fresh interpreter and the pins
# MUST be on disk. If this assert fires, the pair on disk is mixed — that
# state cannot be repaired inside this session. Runtime → Restart session,
# then re-run this cell (the guard above blocks any in-place repair).
state = _probe()
assert state is not None, (
    "torchvision compiled ops failed to load in a fresh interpreter — the classic form is "
    "'RuntimeError: operator torchvision::nms does not exist'. torch/torchvision are ABI-paired; "
    "a mixed pair cannot be repaired in this session. Runtime → Restart session, then re-run this cell."
)
assert state["torch"].startswith(_TORCH_PIN) and state["torchvision"].startswith(_TV_PIN), (
    f"pinned stack not on disk after install: torch {state['torch']} / torchvision {state['torchvision']} "
    f"(expected {_TORCH_PIN}.x / {_TV_PIN}.x)"
)
print(f"compiled-op probe: torch {state['torch']} / torchvision {state['torchvision']} — nms OK")

import importlib.metadata as im
for pkg in ["unsloth", "unsloth_zoo", "transformers", "tokenizers", "trl", "peft",
            "bitsandbytes", "onnx", "onnxruntime", "huggingface_hub"]:
    try:
        print(f"{pkg:18s} {im.version(pkg)}")
    except im.PackageNotFoundError:
        print(f"{pkg:18s} MISSING")
assert im.version("transformers") == "5.2.0", "transformers must be exactly 5.2.0 (unsloth Qwen3.5 tested pin)"
assert im.version("trl") == "0.22.2", "trl must be exactly 0.22.2 (official Qwen3.5 Vision notebook pin)"

# T4 note: Turing (sm_75) has no native bf16 — 'False' here is EXPECTED, not an
# error. Cell 3 auto-selects fp16 on T4 (bf16 only on Ampere+/L4).
# Importing torch is safe NOW: the matched set is fully on disk and verified,
# and nothing in this process loaded a different torch build before it.
import torch
print("bf16 supported:", torch.cuda.is_bf16_supported())


## Cell 2 — Auth & Model Load
Token comes from the `HF_TOKEN` environment variable when driven headlessly (CLI/agent runs),
otherwise a secure `getpass` prompt (never printed) — press Enter to run tokenless. It loads the base
model in **bf16/fp16 at full precision** (no 4-bit: a 0.8B model + LoRA fits comfortably in ~3 GB on the
T4), and attaches LoRA adapters to the **language layers only** — the vision stack is frozen because
Shadow-1-Whisper is a text-contract model.


In [ ]:
# 2 ─ Auth & model load (secure token + LoRA on language layers)
#
# Import unsloth FIRST — before any user interaction. If the environment is
# broken (kernel / torchvision mismatch), the crash must surface here, not
# after the user has already been prompted for a token.
from unsloth import FastVisionModel
import getpass, os
from huggingface_hub import login

# Token acquisition, three contexts:
#   • web UI (interactive): secure getpass prompt — never printed.
#   • CLI / agent run (google-colab-cli exec): pre-set the HF_TOKEN env var.
#   • tokenless dry-run: HF_TOKEN=skip, or Enter at the prompt — the base
#     model is public, so load + training still work; Cells 4/5 then save
#     exports to local folders instead of pushing to the Hub.
_env_token = os.environ.get("HF_TOKEN", "").strip()
if _env_token.lower() in ("skip", "none", "0"):
    HF_TOKEN = None
    print("headless mode (HF_TOKEN=skip) — Cells 4/5 will save exports locally instead of pushing")
elif _env_token:
    HF_TOKEN = _env_token
    login(token=HF_TOKEN)
    print("logged in via HF_TOKEN env var")
else:
    HF_TOKEN = getpass.getpass("🔑 Paste your Hugging Face WRITE token (hf_...) — Enter to skip: ").strip() or None
    if HF_TOKEN:
        assert HF_TOKEN.startswith("hf_"), "That doesn't look like an HF token (should start with 'hf_')"
        login(token=HF_TOKEN)
    else:
        print("no token — continuing tokenless; Cells 4/5 will save exports locally")

BASE_MODEL = "unsloth/Qwen3.5-0.8B"
SEED = 3407

# load_in_4bit=False → full-precision LoRA: ~3 GB VRAM on the T4 (fp16; bf16 on Ampere+)
model, processor = FastVisionModel.from_pretrained(
    BASE_MODEL,
    load_in_4bit=False,
    use_gradient_checkpointing="unsloth",
)

# Attach LoRA — language layers only, vision frozen (text-contract model)
model = FastVisionModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0.0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
    finetune_vision_layers=False,
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
)

FastVisionModel.for_training(model)
model.print_trainable_parameters()

tokenizer = processor.tokenizer if hasattr(processor, "tokenizer") else processor
print("Base model loaded:", BASE_MODEL)
print("dtype:", next(model.parameters()).dtype)


## Cell 3 — Train (SFT on the chat-template)
Loads `shadowtalk_sft.jsonl` (upload it next to the notebook — the cell prompts if missing), renders every
record through `tokenizer.apply_chat_template(..., add_generation_prompt=False)` and trains with
`SFTTrainer` for **3 epochs @ lr 2e-4**, seq len 2048. On a free T4 expect **~25-40 min** for the 620-example run.

In [ ]:
# 3 ─ Train — chat-templated SFT

# Loss-surface note: SFT loss covers all assistant turns, so tool-card turns inside
# multi-turn records must never be reachable as model output. Every card-bearing record
# in shadowtalk_sft.jsonl opens with a request the runtime tool detector
# (useToolOrchestrator.detectToolSync) intercepts before the model is called — calc/
# calculate prefix, bare arithmetic, 'search the web for …', or an explicit /image
# command. The only exceptions are database_query / file_manager, which have no
# auto-detection: their records intentionally teach the honest 'not yet deployed'
# runtime reply instead of a fabricated tool success.
import json, os
from pathlib import Path
from datasets import Dataset
from trl import SFTConfig, SFTTrainer

DATA_FILE = os.environ.get("SHADOWTALK_DATA", "shadowtalk_sft.jsonl")
if not Path(DATA_FILE).exists():
    try:
        from google.colab import files  # browser upload — unavailable under CLI/agent runs
        print("Upload shadowtalk_sft.jsonl now (from training/shadow-1-whisper/)…")
        uploaded = files.upload()
        DATA_FILE = list(uploaded.keys())[0]
    except Exception as e:
        raise RuntimeError(
            f"dataset not found at {DATA_FILE!r} — upload it to /content "
            "(colab upload -s <name> shadowtalk_sft.jsonl /content/shadowtalk_sft.jsonl) "
            "or point SHADOWTALK_DATA at the file"
        ) from e

rows = [json.loads(line) for line in open(DATA_FILE, encoding="utf-8")]
assert rows and all("messages" in r for r in rows), "dataset rows must be {'messages': [...]}"
print(f"loaded {len(rows)} examples")

# Render each record with the model's own chat template.
# add_generation_prompt=False → the assistant turn is TRAINING TARGET, not a prompt prefix.
def render(row):
    return {"text": tokenizer.apply_chat_template(row["messages"], tokenize=False, add_generation_prompt=False)}

dataset = Dataset.from_list([render(r) for r in rows]).shuffle(seed=SEED)
print("sample:\n" + dataset[0]["text"][:600] + "\n…")

MAX_SEQ = 2048
cfg = SFTConfig(
    output_dir="shadow-1-ckpt",
    dataset_text_field="text",
    max_length=MAX_SEQ,                     # trl 0.22.2 renamed max_seq_length → max_length
    num_train_epochs=3,
    learning_rate=2e-4,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,          # effective batch 16
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    fp16=(not torch.cuda.is_bf16_supported()),  # T4 = fp16; A100/L4 = bf16
    bf16=torch.cuda.is_bf16_supported(),
    optim="adamw_8bit",
    weight_decay=0.01,
    logging_steps=5,
    save_strategy="epoch",
    seed=SEED,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=cfg,
)

trainer_stats = trainer.train()

gpu_s = round(trainer_stats.metrics.get("train_runtime", 0), 1)
print(f"trained in {gpu_s}s  |  peak mem ≈ {torch.cuda.max_memory_reserved()/1e9:.2f} GB")


## Cell 4 — Push Merged 16-bit + GGUF q4_k_m
Merges the LoRA adapters back into the base weights, pushes the merged model to
`shadowtalkai/Shadow-1-Whisper-0.8B`, then quantizes to **GGUF `q4_k_m`** and pushes
`shadowtalkai/Shadow-1-Whisper-0.8B-GGUF` (desktop/Tauri runtime). GGUF quantization runs ~8-15 min on the T4.
Any push failure (or a tokenless run) falls back to **saving locally** under `/content/` — retrieve with
`colab download`. Both repos are created **private** — flip them public after you've eyeballed them (see README).


In [ ]:
# 4 ─ Push merged 16-bit + GGUF q4_k_m (or save locally in tokenless mode)
from huggingface_hub import HfApi

# Local export dirs — Cell 5 falls back to MERGED_LOCAL when the merged
# weights are not on the Hub, so the names are part of the contract.
MERGED_LOCAL = "Shadow-1-Whisper-0.8B-merged"
GGUF_LOCAL = "Shadow-1-Whisper-0.8B-GGUF"

if HF_TOKEN:
    api = HfApi(token=HF_TOKEN)
    MERGED_REPO = "shadowtalkai/Shadow-1-Whisper-0.8B"
    GGUF_REPO = "shadowtalkai/Shadow-1-Whisper-0.8B-GGUF"
    for repo in (MERGED_REPO, GGUF_REPO):
        api.create_repo(repo_id=repo, repo_type="model", private=True, exist_ok=True)
    try:
        # 4a — merged 16-bit (the canonical weights + tokenizer + chat template)
        model.push_to_hub_merged(
            MERGED_REPO,
            tokenizer,
            save_method="merged_16bit",
            token=HF_TOKEN,
        )
        print(f"✅ merged 16-bit → https://huggingface.co/{MERGED_REPO}")
    except Exception as e:
        print(f"⚠️ merged push failed ({e.__class__.__name__}: {e}) — saving locally to /content/{MERGED_LOCAL}/")
        model.save_pretrained_merged(MERGED_LOCAL, tokenizer, save_method="merged_16bit")
    try:
        # 4b — GGUF q4_k_m for the Tauri desktop runtime (llama.cpp)
        model.push_to_hub_gguf(
            GGUF_REPO,
            tokenizer,
            quantization_method="q4_k_m",
            token=HF_TOKEN,
        )
        print(f"✅ GGUF q4_k_m → https://huggingface.co/{GGUF_REPO}")
    except Exception as e:
        print(f"⚠️ GGUF push failed ({e.__class__.__name__}: {e}) — saving locally to /content/{GGUF_LOCAL}/")
        model.save_pretrained_gguf(GGUF_LOCAL, tokenizer, quantization_method="q4_k_m")
else:
    print("no HF token — saving exports locally (retrieve with: colab download -s <name> <path>)")
    model.save_pretrained_merged(MERGED_LOCAL, tokenizer, save_method="merged_16bit")
    print(f"✅ merged 16-bit → /content/{MERGED_LOCAL}")
    model.save_pretrained_gguf(GGUF_LOCAL, tokenizer, quantization_method="q4_k_m")
    print(f"✅ GGUF q4_k_m → /content/{GGUF_LOCAL}")


## Cell 5 — Export the WebGPU ONNX package

Transplants the fine-tuned language weights onto the **pinned** reference WebGPU
graph (`onnx-community/Qwen3.5-0.8B-ONNX` @ `8fe9dbe3`), regenerates the quantized
decoders, validates the package end-to-end and publishes it for the browser worker.

1. **Merged weights** — reuses Cell 4's local output only if it is complete, otherwise pulls from the Hub; architecture-checked before use
2. **Transplant** — verified name map + orientation rules (320/320 initializers mapped, 187 transposed, 18 squeezed); hard-fails on ANY unmapped, non-finite or size-ambiguous initializer
3. **q4/q4f16 regenerated** from the transplanted graphs (the reference's quantized files carry BASE weights), with a size-regression check so a no-op quantization cannot ship
4. **Runtime smoke** — every shipped decoder runs one real token on CPU ORT: finite logits required, and a transplanted-vs-base cosine gate catches a silent weight scramble that shape checks cannot
5. **manifest.json** — provenance revisions, per-file sha256, transplant + smoke reports
6. **Publish** — private upload → `shadowtalkai/Shadow-1-Whisper-0.8B-ONNX`, auto-creates the next `v<N>` tag for the worker to pin


In [ ]:
# 5 ─ ONNX transplant: fine-tuned language weights → reference WebGPU graph
#
# Production pipeline: merged weights → PINNED reference graph → transplant
# (verified name map + orientation rules) → regenerate q4/q4f16 → tokenizer
# staging → CPU smoke test → sha256 manifest → private Hub upload + tag.
# Every condition that could indicate a silent weight scramble is a hard assert;
# only mechanical runtime quirks (unexpected graph input signatures) degrade to
# a loud skip instead of blocking the upload.
import glob, hashlib, json, os, re, shutil, time
from pathlib import Path

import numpy as np
import onnx
from onnx import TensorProto, numpy_helper
from huggingface_hub import HfApi, snapshot_download

# ── contract with cells 2/4 (these names are part of the kit) ────────────────
REF_REPO = "onnx-community/Qwen3.5-0.8B-ONNX"
REF_REVISION = "8fe9dbe3fb39fe902e18df9be84b6a0ac77ae0f6"  # pinned 2026-11-28 — the transplant map below was verified against exactly this export
MERGED_REPO = "shadowtalkai/Shadow-1-Whisper-0.8B"   # same constant as cell 4
MERGED_DIR = "Shadow-1-Whisper-0.8B-merged"           # local copy of cell-4 output
ONNX_REPO = "shadowtalkai/Shadow-1-Whisper-0.8B-ONNX"

# defensive: cell 5 must not depend on cell 4's locals — re-resolve the token
HF_TOKEN = globals().get("HF_TOKEN") or os.environ.get("HF_TOKEN")
api = HfApi(token=HF_TOKEN)

# 5.0 — merged weights (cell-4 output). A local dir wins ONLY if it is complete;
# a stale or partial dir from an earlier session must never masquerade as the result.
def _merged_dir_ready(d):
    p = Path(d)
    return p.exists() and any(p.glob("*.safetensors")) and (p / "config.json").exists()

if not _merged_dir_ready(MERGED_DIR):
    # cell 4 pushed to the Hub; pull the same merged weights back down.
    # Tokenless runs have no Hub access to a private repo — they must come
    # here via Cell 4's local-save fallback instead.
    if HF_TOKEN is None:
        raise RuntimeError(
            f"no HF token and no complete local merged weights at {MERGED_DIR}/ — run Cell 4 first "
            "(tokenless mode saves them locally), or set HF_TOKEN"
        )
    print(f"pulling merged weights from {MERGED_REPO} …")
    MERGED_DIR = str(snapshot_download(MERGED_REPO, token=HF_TOKEN,
                                       local_dir="Shadow-1-Whisper-0.8B-merged"))
merged_cfg = json.loads((Path(MERGED_DIR) / "config.json").read_text())
assert any("Qwen3_5" in a for a in merged_cfg.get("architectures", [])), (
    f"merged config architectures = {merged_cfg.get('architectures')} — not a Qwen3.5 "
    "checkpoint; wrong MERGED_DIR?"
)
merged_sha = None
if HF_TOKEN:
    try:
        merged_sha = api.model_info(MERGED_REPO, token=HF_TOKEN).sha
    except Exception:
        pass
print("merged weights ready:", MERGED_DIR,
      f"(hub revision {str(merged_sha)[:8]})" if merged_sha else "(local build)")

# 5.1 — reference graph → working dir (PINNED revision; see REF_REVISION above)
work = Path("Shadow-1-Whisper-0.8B-ONNX-work")
shutil.rmtree(work, ignore_errors=True)
try:
    ref_dir = Path(snapshot_download(REF_REPO, revision=REF_REVISION))
    ref_sha_used = REF_REVISION
except Exception as e:
    print(f"⚠️ pinned reference revision unavailable ({e.__class__.__name__}) — "
          "resolving HEAD; the transplant gates below must re-verify the map")
    ref_dir = Path(snapshot_download(REF_REPO))
    ref_sha_used = api.model_info(REF_REPO).sha
shutil.copytree(ref_dir, work, dirs_exist_ok=True, ignore=shutil.ignore_patterns(".cache"))
print("reference graph files:", len(list(work.rglob("*.onnx"))))

# 5.2 — merged state dict (safetensors shards)
from safetensors.torch import load_file
state = {}
for shard in sorted(glob.glob(f"{MERGED_DIR}/*.safetensors")):
    state.update(load_file(shard))
print(f"merged state dict: {len(state)} tensors")
EXPECTED_TENSORS = 488   # verified baseline for Qwen3.5-0.8B
assert len(state) >= 300, f"only {len(state)} tensors in {MERGED_DIR} — incomplete merged weights?"
if len(state) != EXPECTED_TENSORS:
    print(f"⚠️ {len(state)} tensors ≠ verified baseline {EXPECTED_TENSORS} — read the transplant report closely")

# The transplant rules below were verified offline against the REAL reference graph
# (decoder_model_merged_fp16.onnx @ 8fe9dbe3: 397 initializers) and the REAL base
# state dict (488 tensors). Two structural facts drive everything:
#   1. NAMING — optimum exported fused names like
#      model.layers.3.attn.q_proj.MatMul.weight, while transformers v5 stores
#      model.language_model.layers.3.self_attn.q_proj.weight. graph_to_hf() is the
#      exact family map (attn→self_attn, gdn→linear_attn, fused-infix strip,
#      tied lm_head→embed_tokens, layer-24 final norm→model.language_model.norm,
#      q/k_norm.layernorm→q/k_norm).
#   2. ORIENTATION — the exporter folded the Linear transpose: graph weight dims
#      are the REVERSE of the HF (out,in) shapes (graph [1024,4096] vs HF
#      (4096,1024)). The only correct transform is a true .T — a same-size reshape
#      would silently scramble every weight.
# LoRA touches only the decoder's Linear projections, but mapping + transplanting
# EVERYTHING that maps is still correct: untouched tensors write identical values.
# The reference's embed_tokens*/vision_encoder* graph FILES stay untouched for the
# same reason (their weights are byte-identical to base).

def graph_to_hf(init_name: str):
    """Exact structural mapping from the reference graph's initializer names to
    the transformers-v5 state dict (verified offline; see the block comment)."""
    n = init_name
    if n.startswith("/model/constants/"):   # exporter-baked constants (shapes, eps, limits)
        return None
    if n == "model.inv_freq":                # RoPE buffer — config-derived
        return None
    if n.endswith("A_neg_exp"):              # derived: exp(-A_log); A_log is not LoRA-targeted
        return None
    n = re.sub(r"\.(MatMul|Gemm|Transpose)\.weight$", ".weight", n)  # optimum fused-op infix
    n = n.replace("conv1d.weight_3d", "conv1d.weight")  # exporter's 3-D view of the same conv weight
    if n == "lm_head.weight":                # tied to the embedding in Qwen3.5
        return "model.language_model.embed_tokens.weight"
    if n.startswith("model.layers."):
        if n.endswith("final_norm_layernorm.weight"):   # graph names it layer 24; HF: model norm
            return "model.language_model.norm.weight"
        n = "model.language_model.layers." + n[len("model.layers."):]
        n = n.replace(".attn.", ".self_attn.")
        n = n.replace(".gdn.", ".linear_attn.")
        n = n.replace("_norm.layernorm.weight", "_norm.weight")
    return n

def find_weight(init_name: str):
    """Initializer name → state-dict key. Exact structural map first, then raw
    name / drifted variants, then a UNIQUE-suffix fallback. Ambiguous suffix
    matches return None — never guess a weight."""
    hf = graph_to_hf(init_name)
    if hf is None:
        return None
    candidates = [hf, init_name,
                  init_name.removeprefix("module."),
                  init_name.split("onnx::")[-1],
                  init_name.replace("onnx::", "").replace("~", ".")]
    for cand in candidates:
        if cand in state:
            return cand
    matches = {sk for sk in state
               if len(init_name) >= 8 and (init_name.endswith(sk) or sk.endswith(init_name))}
    return next(iter(matches)) if len(matches) == 1 else None

# non-persistent / derived buffers — baked from config or recomputed by the
# exporter; they must never be transplanted (and never count as missing)
NO_TRANSPLANT_INITS = re.compile(r"(inv_freq|rotary_emb|position_ids|cached_|A_neg_exp)", re.I)

TRANSPLANT_FILES = ["onnx/decoder_model_merged.onnx", "onnx/decoder_model_merged_fp16.onnx"]
MIN_COVERAGE = 0.70   # belt-and-braces floor; the hard gates below demand far more
report = {}
for rel in TRANSPLANT_FILES:
    onnx_path = work / rel
    m = onnx.load(str(onnx_path), load_external_data=True)
    matched, missing, mismatched, ambiguous, nonfinite = 0, [], [], [], []
    transposed, squeezed, skipped = 0, 0, []
    total = len(m.graph.initializer)
    for init in m.graph.initializer:
        if init.name.startswith("/model/constants/") or NO_TRANSPLANT_INITS.search(init.name):
            skipped.append(init.name)
            continue
        sk = find_weight(init.name)
        if sk is None:
            missing.append(init.name)
            continue
        arr = state[sk].detach().cpu().numpy()
        dims = list(init.dims)
        if dims and list(arr.shape) != dims:
            if arr.ndim == 2 and dims == list(arr.shape)[::-1]:
                arr = arr.T.copy()        # exporter folded the Linear transpose — true .T
                transposed += 1
            elif arr.ndim == 3 and arr.shape[1] == 1 and dims == [arr.shape[0], arr.shape[2]]:
                arr = arr.reshape(dims)   # conv1d: HF (C,1,K) → graph (C,K) — lossless axis-1 squeeze
                squeezed += 1
            elif arr.size == int(np.prod(dims)):
                ambiguous.append((init.name, list(arr.shape), dims))
                continue
            else:
                mismatched.append(init.name)
                continue
        # keep the graph's declared dtype (fp16 graphs stay fp16)
        if init.data_type == TensorProto.FLOAT16:
            arr = arr.astype(np.float16)
        elif init.data_type == TensorProto.FLOAT:
            arr = arr.astype(np.float32)
        if not np.isfinite(arr).all():
            nonfinite.append(init.name)
            continue
        init.CopyFrom(numpy_helper.from_array(arr, name=init.name))
        matched += 1
    effective = total - len(skipped)
    coverage = matched / max(1, effective)
    report[rel] = {
        "initializers": total, "matched": matched, "transposed": transposed,
        "squeezed": squeezed, "missing": len(missing),
        "shape_mismatch": len(mismatched), "size_ambiguous": len(ambiguous),
        "buffers_skipped": len(skipped), "nonfinite": len(nonfinite),
        "coverage": round(coverage, 3),
    }
    if missing:
        print(f"  missing in {rel}: {missing[:12]}{' …' if len(missing) > 12 else ''}")
    # ── production gates — the pinned reference was verified offline at 320/320
    # mapped (187 transposed, 18 squeezed, 77 exporter buffers skipped):
    assert not missing, (
        f"{rel}: {len(missing)} initializers do not map into the merged state dict "
        f"({missing[:8]}) — the reference export drifted from REF_REVISION; extend "
        "graph_to_hf() or re-pin. Never ship a partial transplant."
    )
    assert coverage >= MIN_COVERAGE, (
        f"{rel}: transplant coverage {coverage:.0%} < {MIN_COVERAGE:.0%} — inspect the "
        "report and extend graph_to_hf() before shipping."
    )
    assert not ambiguous, (
        f"{rel}: size-equal shape mismatches {ambiguous[:6]} — a blind reshape would "
        "silently scramble transposed weights; extend graph_to_hf() instead."
    )
    assert not mismatched, f"{rel}: true shape mismatches {mismatched[:6]}"
    assert not nonfinite, f"{rel}: non-finite merged values in {nonfinite[:8]} — corrupted source shards?"
    assert transposed >= 100 and squeezed >= 10, (
        f"{rel}: orientation rules fired only {transposed}x transpose / {squeezed}x squeeze "
        "(verified baseline 187/18) — the reference naming changed; re-verify before shipping."
    )
    onnx.save_model(
        m, str(onnx_path),
        save_as_external_data=True,
        all_tensors_to_one_file=True,
        location=onnx_path.name + ".data",   # relative to the model file's own dir
        size_threshold=1024,
    )

print(json.dumps(report, indent=2))

# 5.4 — regenerate the QUANTIZED decoder variants from the transplanted graphs.
# The reference's _q4/_q4f16/_quantized files carry BASE weights (int4/int8-packed
# initializers a float transplant can never reach): shipping them untouched would
# silently serve the BASE model to every WebGPU user selecting dtype "q4".
# ORT >=1.20 renamed the module AND the class (MatMul4BitsQuantizer ->
# MatMulNBitsQuantizer, new dep: onnx_ir); older ORT keeps matmul_4bits_quantizer.
try:
    from onnxruntime.quantization.matmul_nbits_quantizer import MatMulNBitsQuantizer as NBitsQuantizer
except ModuleNotFoundError:
    try:
        from onnxruntime.quantization.matmul_4bits_quantizer import MatMul4BitsQuantizer as NBitsQuantizer
    except ModuleNotFoundError as e:
        raise SystemExit('onnxruntime n-bits quantizer unavailable — `pip install onnx_ir onnxruntime` and re-run this cell') from e

def quantize_nbits(src: Path, dst: Path):
    m = onnx.load(str(src), load_external_data=True)
    q = NBitsQuantizer(m, block_size=32, is_symmetric=True,
                       op_types_to_quantize=["MatMul", "Gemm"])
    q.process()
    proto = q.model.model if hasattr(q.model, "model") else q.model
    onnx.save_model(proto, str(dst),
                    save_as_external_data=True, all_tensors_to_one_file=True,
                    location=dst.name + ".data", size_threshold=1024)

quantize_nbits(work / "onnx/decoder_model_merged.onnx",
               work / "onnx/decoder_model_merged_q4.onnx")
print("regenerated q4 decoder from transplanted fp32 graph")
try:
    quantize_nbits(work / "onnx/decoder_model_merged_fp16.onnx",
                   work / "onnx/decoder_model_merged_q4f16.onnx")
    print("regenerated q4f16 decoder from transplanted fp16 graph")
except Exception as e:
    # loud fallback: delete the variant instead of shipping stale base weights
    print("! q4f16 regeneration failed — deleting the variant:", e)
    for p in work.glob("onnx/decoder_model_merged_q4f16*"):
        p.unlink()

# q4 must actually be smaller — a no-op quantization would ship fp32-sized "q4"
_f32_data = work / "onnx/decoder_model_merged.onnx.data"
_q4_data = work / "onnx/decoder_model_merged_q4.onnx.data"
if _f32_data.exists() and _q4_data.exists():
    _fb, _qb = _f32_data.stat().st_size, _q4_data.stat().st_size
    assert _qb < _fb, "q4 external data is not smaller than fp32 — quantization was a no-op"
    print(f"q4 packing: {_fb >> 20} MB → {_qb >> 20} MB ({_qb * 100 // max(_fb, 1)}% of fp32)")

# stale int8 variant: not regenerable with ORT dynamic quantization one-to-one,
# and unused by the documented worker contract (dtype q4) — remove, don't ship base.
for p in sorted(work.glob("onnx/decoder_model_merged_quantized*")):
    p.unlink()
    print("removed stale base-weight int8 variant:", p.name)

# superseded external-data blobs (ref naming: .onnx_data / .onnx_data_1)
for name in ["decoder_model_merged", "decoder_model_merged_fp16",
             "decoder_model_merged_q4", "decoder_model_merged_q4f16",
             "decoder_model_merged_quantized"]:
    for p in work.glob(f"onnx/{name}.onnx_data*"):
        p.unlink()

# 5.5 — tokenizer + configs from the merged model (chat_template included)
TOKEN_FILES = ["tokenizer.json", "tokenizer_config.json", "special_tokens_map.json",
               "generation_config.json", "chat_template.jinja", "added_tokens.json"]
for name in TOKEN_FILES:
    src = Path(MERGED_DIR) / name
    if src.exists():
        shutil.copy(src, work / name)
# older transformers keep the template inside tokenizer_config.json
tc_path = work / "tokenizer_config.json"
if tc_path.exists() and not (work / "chat_template.jinja").exists():
    cfg_json = json.loads(tc_path.read_text())
    if "chat_template" in cfg_json:
        (work / "chat_template.jinja").write_text(cfg_json["chat_template"])
print("tokenizer + configs staged")
# the ROOT config.json must stay the REFERENCE's (it carries the exported graph's
# metadata and is what transformers.js reads); the merged config.json is only
# arch-checked in 5.0.
assert (work / "config.json").exists(), "reference config.json missing — transformers.js cannot load this package"
for _opt in ["preprocessor_config.json", "processor_config.json"]:
    if not (work / _opt).exists():
        print(f"⚠️ {_opt} not in the reference export — AutoProcessor may need it for vision inputs")

# 5.6 — onnxruntime shape verification: every decoder graph we SHIP must load and
# expose the documented IO (float transplants + regenerated q4/q4f16).
import onnxruntime as ort
print("ORT providers:", ort.get_available_providers())
main_models = sorted(work.glob("onnx/decoder_model_merged*.onnx"))
assert main_models, "no decoder graphs left to verify — regeneration failed?"
for p in main_models:
    sess = ort.InferenceSession(str(p), providers=["CPUExecutionProvider"])
    ins = [(i.name, i.shape) for i in sess.get_inputs()]
    outs = [(o.name, o.shape) for o in sess.get_outputs()]
    print(f"✓ {p.relative_to(work)}  inputs={ins[:3]}… outputs={outs[:2]}…")
    del sess

# 5.7 — runtime smoke: run every shipped decoder for ONE real token on CPU ORT.
# Shape checks cannot catch a scrambled transplant; a live forward pass proves
# the graph computes finite logits AND that the fine-tune actually moved them.
def _build_feeds(sess, token_id, mode):
    feeds = {}
    for i in sess.get_inputs():
        name, shp, itype = i.name, list(i.shape), i.type
        np_dt = np.int64 if "int" in itype else np.float32
        r = lambda d: d if isinstance(d, int) else 1
        is_core = ("input_ids" in name or "mask" in name or "position" in name
                   or "past" in name or "cache" in name)
        if mode == "core" and not is_core:
            continue
        if mode == "no_embeds" and "inputs_embeds" in name:
            continue
        if "input_ids" in name:
            feeds[name] = np.full([1, 1], token_id, dtype=np.int64)
        elif "position" in name:
            feeds[name] = np.zeros([1, 1], dtype=np.int64)
        elif "mask" in name:
            feeds[name] = np.ones([1, 1], dtype=np.int64)
        elif "past" in name or "cache" in name:
            dims = [r(d) for d in shp]
            if len(dims) >= 3:
                dims[2] = 0            # KV sequence axis → empty past (prefill step 1)
            feeds[name] = np.zeros(dims, dtype=np_dt)
        else:
            feeds[name] = np.zeros([r(d) for d in shp], dtype=np_dt)
    return feeds

def _run_once(sess, token_id):
    last_err = None
    for mode in ("core", "no_embeds", "all"):
        try:
            return sess.run(None, _build_feeds(sess, token_id, mode)), mode
        except Exception as e:
            last_err = e
    raise last_err

def _logits(outs):
    arr = max(outs, key=lambda o: o.size)   # logits, not the KV-cache outputs
    return np.asarray(arr, dtype=np.float64).reshape(-1)

try:
    _vocab = json.loads((work / "tokenizer.json").read_text())["model"]["vocab"]
    _token_id = int(_vocab.get("Hello", 0))
except Exception:
    _token_id = 0
smoke = {}
for p in main_models:
    sess = None
    run_err = None
    try:
        sess = ort.InferenceSession(str(p), providers=["CPUExecutionProvider"])
        out, mode = _run_once(sess, _token_id)
        logits = _logits(out)
    except Exception as e:
        run_err = f"{e.__class__.__name__}: {str(e)[:200]}"
    if sess is not None:
        del sess
    if run_err is not None:
        smoke[p.name] = {"status": "skipped", "reason": run_err}
        print(f"⚠️ smoke skipped for {p.name}: {run_err}")
        continue
    assert np.isfinite(logits).all(), f"{p.name}: NON-FINITE logits — the graph is corrupt"
    entry = {"status": "ok", "feed_mode": mode, "logits_dim": int(logits.shape[0])}
    if p.name == "decoder_model_merged.onnx" and (ref_dir / "onnx" / p.name).exists():
        base_sess = ort.InferenceSession(str(ref_dir / "onnx" / p.name),
                                         providers=["CPUExecutionProvider"])
        base_logits = _logits(_run_once(base_sess, _token_id)[0])
        del base_sess
        assert np.isfinite(base_logits).all(), "BASE graph produced non-finite logits?!"
        denom = float(np.linalg.norm(logits) * np.linalg.norm(base_logits))
        cos = float(np.dot(logits, base_logits) / max(denom, 1e-12))
        delta = float(np.abs(logits - base_logits).max())
        entry.update({"cosine_vs_base": round(cos, 4), "max_logit_delta": round(delta, 6)})
        assert cos >= 0.5, (
            f"transplanted logits diverge from base (cosine {cos:.3f} < 0.5) — the transplant "
            "scrambled the weights; do NOT ship this package"
        )
        if delta <= 1e-4:
            entry["status"] = "ok_no_delta"
            print(f"⚠️ {p.name}: transplanted logits ≈ base (max delta {delta:.2e}) — "
                  "fine-tune deltas may be tiny; verify LoRA actually merged in Cell 4")
    print(f"✓ smoke {p.name}: {entry['status']} (cosine vs base {entry.get('cosine_vs_base', 'n/a')})")
    smoke[p.name] = entry

# 5.8 — package README + manifest.json: provenance, integrity, worker wiring
(work / "README.md").write_text("\n".join([
    "# Shadow-1-Whisper-0.8B — WebGPU ONNX package",
    "",
    "Fine-tuned Qwen3.5-0.8B language weights transplanted onto the reference",
    f"WebGPU graph `{REF_REPO}` @ `{ref_sha_used[:8]}`.",
    "",
    f"- merged 16-bit source: {MERGED_REPO}" + (f" @ {merged_sha[:8]}" if merged_sha else " (local build)"),
    "- language decoders (onnx/decoder_model_merged*.onnx): transplanted + regenerated q4/q4f16",
    "- embed_tokens* / vision_encoder*: reference graphs, byte-identical to base (not LoRA-targeted)",
    '- browser runtime: @huggingface/transformers v4+, dtype "q4", device "webgpu"',
    "  (temperature 0.7 / top_p 0.8 / top_k 20 / presence_penalty 1.5)",
    "- integrity + provenance: manifest.json (per-file sha256, transplant stats, smoke report)",
]) + "\n")
def _sha256(p, chunk=1 << 20):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        while True:
            b = f.read(chunk)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

manifest = {
    "generated_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "merged": {"repo": MERGED_REPO, "revision": merged_sha, "tensors": len(state),
               "local_dir": str(MERGED_DIR)},
    "reference": {"repo": REF_REPO, "revision": ref_sha_used},
    "transplant": report,
    "runtime_smoke": smoke,
    "tooling": {"onnx": onnx.__version__, "onnxruntime": ort.__version__},
    "files": {},
}
for p in sorted(work.rglob("*")):
    if p.is_file() and (p.suffix in {".onnx", ".json", ".jinja", ".md"}
                        or ".onnx_data" in p.name or p.name.endswith(".data")):
        manifest["files"][str(p.relative_to(work)).replace("\\", "/")] = {
            "bytes": p.stat().st_size, "sha256": _sha256(p)}
(work / "manifest.json").write_text(json.dumps(manifest, indent=2))
print(f"manifest: {len(manifest['files'])} files hashed "
      f"({sum(v['bytes'] for v in manifest['files'].values()) >> 20} MB)")

# 5.9 — publish (private; flip public after inspection). Tokenless runs keep the
# package local instead of failing the whole cell at the finish line.
uploaded_tag = None
try:
    api.create_repo(ONNX_REPO, repo_type="model", private=True, exist_ok=True)
    api.upload_folder(folder_path=str(work), repo_id=ONNX_REPO, repo_type="model",
                      commit_message=f"Shadow-1-Whisper: transplant onto pinned reference {ref_sha_used[:8]} + tokenizer + manifest")
    print(f"✅ ONNX package → https://huggingface.co/{ONNX_REPO}")
    try:
        _tags = [t.name for t in (api.list_repo_tags(ONNX_REPO) or [])]
        _n = 1 + len([t for t in _tags if t.startswith("v") and t[1:].isdigit()])
        api.create_tag(ONNX_REPO, repo_type="model", tag=f"v{_n}")
        uploaded_tag = f"v{_n}"
        print(f"🏷 tagged {uploaded_tag} — pin the worker MODEL_ID to {ONNX_REPO}@{uploaded_tag}")
    except Exception as te:
        print(f"⚠️ tag skipped ({te.__class__.__name__}) — pin via manifest.json revisions instead")
except Exception as e:
    print(f"⚠️ upload skipped ({e.__class__.__name__}: {e}) — ONNX package kept at /content/{work.name}")

print()
print("NEXT — flip the repos PUBLIC after inspection, then wire the web runtime:")
print("  @huggingface/transformers v4+ — AutoProcessor + Qwen3_5ForConditionalGeneration")
print(f'  from "{ONNX_REPO}"' + (f" @ {uploaded_tag}" if uploaded_tag else " (pin the commit recorded in manifest.json)"))
print('  dtype: "q4" · device: "webgpu" · temperature 0.7 / top_p 0.8 / top_k 20 / presence_penalty 1.5')
print("  (README.md — section 5, 'Wire the ONNX package into ShadowTalk's WebGPU worker')")
